In [118]:
import pandas as pd
import matplotlib.pyplot as plt

In [119]:
# Load dataset from csv
dataset = pd.read_csv("./datasets/PreProcessedDataset.csv")

dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Columns: 121 entries, pCR (outcome) to IsoForest_Outlier
dtypes: float64(120), int64(1)
memory usage: 378.2 KB


# Feature Selection / Dimensionality reduction
### Note: specification says to keep the following features: 'ER', 'HER2' and 'Gene' 

Feature Selection
- Wrapper methods: forward feature selection, recursive feature elimination
- Embedded methods: LASSO, Decision Trees
- Filter based methods: t-tests, chi square, univariate ANOVA, correlation analysis, idk what else slides say


Dimensionality Reduction
- Includes: PCA, LDA, manifold learning
- Is manifold learning even applicable? 


First do Correlation analysis for visualisation: https://www.geeksforgeeks.org/machine-learning/data-preprocessing-machine-learning-python/#:~:text=Step%205%3A%20Correlation%20Analysis

- Also other methods in lab probably

In [120]:
# Preserve important features
dataset_preserve = dataset['Gene','ER','HER2']
dataset = dataset.drop(labels=['Gene', 'ER', 'HER2'], axis=1)


# Split data into features and outcomes

# Outcomes
dataset_y_clf = dataset['pCR (outcome)']
datasey_y_reg = dataset['RelapseFreeSurvival (outcome)']

# Features
dataset = dataset.drop(['pCR (outcome)', 'RelapseFreeSurvival (outcome)'], axis=1)

dataset

KeyError: ('Gene', 'ER', 'HER2')

### Feature Selection
Feature Selection aims to create a subset of the original features optimal for the model building process. This is split into 3 categories.
- Wrapper Methods
- Embedded Methods
- Filter-based Methods

##### Wrapper Methods
These likely take too long as they iterate through different feature permutations and evaluate them using an evaluation metric. Possibly apply after another method that has already reduced the number of features.

Options:
1. Forward Feature Selection
2. Recursive Feature Elimination



##### Embedded Methods
Embed Feature Selection into the model process. Should probably use in the ML Method Development process.

Options:
1. LASSO (Regression)
2. Decision Trees / Random Forest

##### Filter-based Methods
These methods are dependent on the datatypes of features (Xs) and outcomes (ys). Feature datatypes are both categorical and continuous, meaning different methods would have to be applied based on this. The classification outcome, 'pCR (outcome)', is categorical data; meanwhile the regression outcome, 'RelapseFreeSurvival (outcome)', is continuous data.

Options:

- t-test
- chi-square
- ANOVA
- Correlation Analysis

t-test requirements:
- Gaussian Distribution: data is close to gaussian distribution
- Binary Features: data contains binary features
- Continuous Outcome: RelapseFreeSurvival (outcome) is the continuous outcome

Conclusion: Applicable, features would need to be split

chi-square requirements:
- Categorical Features: dataset contains categorical features
- Categorical Outcome: pCR (outcome) is the categorical outcome

Conclusion: Applicable, features would need to be split

ANOVA requirements:
- Continuous Features: dataset contains continuous features
- Categorical Outcome: pCR (outcome) is the categorical outome

Conclusion: Applicable, features would need to be split

Correlation Analysis requirements:
- Continuous Feature: dataset contains continuous features
- Continuous Outcome: RelapseFreeSurvival (outcome) is the continuous outcome

Conclusion: Applicable, features would need to be split

### Dimensionality Reduction
Reduces the dimension of the data by generating new features from the old ones. This reduces interpretability, as new features are not the same as the old ones, hence we lose some understanding of the dataset

##### PCA (Principal Component Analysis)
This unsupervised learning method fits data to a new pair of axes so that it can be split using less features. This is done by calculating the mean (origin of new axes), covariance matrix (ellipse that contains the data), eigenvalues and eigenvectors of the covariance matrix (pair of new axes). 

scikit-learn pca has the following approach:
- n_components is the desired output feature total

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=10)
Xs_pca = pca.fit_transform(dataset)
PCA_df = pd.DataFrame(Xs_pca, columns=['PCA_'+str(i+1) for i in range(Xs_pca.shape[1])])

PCA_df

,PCA_1,PCA_2,PCA_3,PCA_4,PCA_5,PCA_6,PCA_7,PCA_8,PCA_9,PCA_10
0,-2.730928,0.957925,0.544377,-0.276261,0.138983,0.110791,-0.494491,0.276201,-0.392768,0.494865
1,0.078035,1.070808,-0.604498,0.388219,1.559529,-0.001648,-0.499613,0.112747,0.438481,0.046987
2,0.117276,0.095397,-1.122872,0.448856,-1.191866,-0.818262,0.016601,-0.041815,0.091951,-0.048208
3,0.459781,2.191060,-0.242672,0.297955,-0.215352,-0.309631,-0.353833,0.553392,0.041214,0.206103
4,0.141832,-0.465131,-0.852125,0.478594,-0.154532,-0.978325,0.193292,-0.034142,-0.015607,0.144321
...,...,...,...,...,...,...,...,...,...,...
395,1.378182,0.940898,0.579532,0.256832,0.519258,0.263881,0.421859,0.558321,-0.727600,-0.292396
396,-2.462868,0.965885,0.391264,-0.232982,0.080811,0.328926,0.055315,-0.421186,0.283966,-0.459734
397,0.429920,-0.099536,-1.473312,0.190208,-1.246014,0.945758,-0.021391,-0.433941,-0.274998,-0.516749
398,-0.237377,1.378835,0.569356,-0.190121,1.047362,-0.273595,-0.345321,-0.202947,-0.407742,0.310307


Plotting PCA Features

In [ ]:
plt.figure(figsize=(6,6))
plt.plot(PCA_df['PCA_1'][dataset['pCR (outcome)'] == 1], PCA_df['PCA_2'][dataset['pCR (outcome)'] == 1], 'ro', alpha = 0.7, markeredgecolor = 'k')
plt.plot(PCA_df['PCA_1'][dataset['pCR (outcome)'] == 0], PCA_df['PCA_2'][dataset['pCR (outcome)'] == 0], 'bo', alpha = 0.7, markeredgecolor = 'k')
plt.xlabel('PCA_1')
plt.ylabel('PCA_2')
plt.legend(['pCR','not pCR'])

KeyError: 'pCR (outcome)'

<Figure size 600x600 with 0 Axes>

##### LDA (Linear Discriminant Analysis)
This supervised learning method projects data to a new pair of axes that helps split it into different classes. This is done by maximising the variance between data of different classes to separate them, and minimising the mean to keep datapoints within the same class together

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

lda = LDA()
dataset_features = dataset[]
lda.fit(X=dataset_features,y=)


SyntaxError: invalid syntax (3341178497.py, line 4)